Score commercial auto policies for expected claim risk
Fits a model on historical data and scores the current period

Data:
train.csv  - policies 2020-2022, includes claim outcomes
score.csv  - policies 2023-2024, no outcomes (to be scored)

# Commercial Auto Risk Scoring

## Imports and data loading

In [0]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [0]:
DATA_DIR = Path('.')
train_raw = pd.read_csv(DATA_DIR / 'data' / 'train.csv')
score_raw = pd.read_csv(DATA_DIR / 'data' / 'score.csv')

## Original files and missing information

Start with the file dimensions, example rows, and column inventory. The inventory distinguishes missing values in an available column from a column absent from scoring

In [0]:
print(f'Training: {train_raw.shape[0]:,} rows, {train_raw.shape[1]} columns')
print(f'Scoring: {score_raw.shape[0]:,} rows, {score_raw.shape[1]} columns')
print('Training columns absent from scoring:', sorted(set(train_raw) - set(score_raw)))

In [0]:
train_raw.head()

In [0]:
score_raw.head()

In [0]:
checks = pd.DataFrame({
    'Type': train_raw.dtypes,
    'Missing in train': train_raw.isna().sum(),
    'Missing in train, %': train_raw.isna().mean().mul(100).round(2),
    'Missing in score': score_raw.isna().sum().reindex(train_raw.columns),
    'Missing in score, %': score_raw.isna().mean().mul(100).reindex(train_raw.columns).round(2),
    'Score present': train_raw.columns.isin(score_raw.columns),
})
checks

Training has 15,075 raw rows and 31 columns; scoring has 3,000 rows and 28 columns. Scoring lacks `claim_coun` and `total_loss_amount` because outcomes are unknown. It also lacks `has_safety_program`, so that predictor cannot be used with the supplied scoring file.

## Policy identifiers, categories, and duplicate copies

The observation is a policy coverage line, identified by policy_id. insured_id identifies the company and can appear on several coverage lines or periods; repeated insured IDs are not duplicate rows.

In [0]:
for name, data in [('Train', train_raw), ('Score', score_raw)]:
    print(
        name,
        '| extra exact copies:', data.duplicated().sum(),
        '| repeated policy IDs:', data.policy_id.duplicated().sum(),
        '| distinct insureds:', data.insured_id.nunique()
    )

In [0]:
print('Policy IDs shared across files:', len(set(train_raw['policy_id']) & set(score_raw['policy_id'])))
print('Insured IDs shared across files:', len(set(train_raw['insured_id']) & set(score_raw['insured_id'])))

### Categorical values and normalization

In [0]:
def check_categorical(col):
    counts = pd.DataFrame({
        'Train': train_raw[col].fillna('Missing').value_counts(),
        'Score': score_raw[col].fillna('Missing').value_counts(),
    }).fillna(0).astype(int)

    new_values = set(score_raw[col].dropna()) - set(train_raw[col].dropna())
    if new_values:
        print('Values appearing only in scoring:', sorted(new_values))
    
    return counts


In [0]:
check_categorical('coverage_type')

In [0]:
check_categorical('business_type')

In [0]:
check_categorical('state')

In [0]:
check_categorical('payment_frequency')

In [0]:
train = train_raw.copy()
score = score_raw.copy()
for data in [train, score]:
    data['business_type'] = data.business_type.str.strip().str.lower()

In [0]:
for name, data in [('Train', train), ('Score', score)]:
    repeated = data.loc[data['policy_id'].duplicated(keep=False)]
    differences = repeated.groupby('policy_id').nunique(dropna=False)
    conflicts = differences.gt(1).any(axis=1)
    print(
        name,
        '| extra copies after normalization:', data.duplicated().sum(),
        '| conflicting policy IDs:', conflicts.sum()
    )
    if conflicts.any():
        display(repeated.loc[repeated['policy_id'].isin(conflicts.index[conflicts])])

train = train.drop_duplicates().copy()
score = score.drop_duplicates().copy()
print(f'After deduplication: {len(train):,} training lines; {len(score):,} scoring lines')

Remove the 75 extra training copies after normalization. Of these, 42 were already exact copies and another 33 differed only in business-type case. There are no conflicting policy IDs; scoring has no copies to remove.

APD and AL each have 7,500 deduplicated training lines. Both files contain the same 12 state codes and payment schedules. Training has 8 agriculture lines; scoring has 60 rideshare lines. These labels are outside the dictionary's example list, but the file does not identify a corrected category. Keep them and allow the encoder to handle new values.
The plot compares each business type's share of its own file.

In [0]:
business_mix = pd.DataFrame({
    name: data['business_type'].value_counts().div(len(data)).mul(100)
    for name, data in [('Train', train), ('Score', score)]
}).fillna(0).sort_values('Train')

fig, ax = plt.subplots(figsize=(10, 5), dpi=200)
business_mix.plot.barh(figsize=(9, 4.5), ax=ax)
for bars in ax.containers:
    ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=8)

ax.set(
    title='Business types of policies',
    xlabel='Policy lines (%)',
    ylabel='',
    xlim=(0, 37)
)
ax.legend(frameon=False)
ax.grid(axis='x', alpha=0.2)

plt.tight_layout()
plt.show()
plt.close()

The main business mix is similar. Rideshare makes up 2% of scoring, while agriculture is a very small training category.

## Numeric predictors and values to review

Select common numeric inputs and exclude claim outcomes and current-period claim/payment information before checking their ranges.

In [0]:
excluded_numeric = [
    'claim_count', 'total_loss_amount', 'has_safety_program',
    'late_payment_count', 'claim_paid_amount_current_period', 'days_to_first_claim_report',
    'had_claim', 'snapshot_year', 'policy_duration_days', 'snapshot_delay_days'
]
numeric_features = [
    col for col in train.select_dtypes(include='number').columns
    if col in score.columns and col not in excluded_numeric
]
numeric_features

In [0]:
numeric_issues = {}
for name, data in [('Train', train), ('Score', score)]:
    numeric_issues[name] = pd.DataFrame({
        'Negative underwriting value': data[numeric_features].lt(0).any(axis=1),
        'Heavy vehicles exceed fleet': data['num_heavy_vehicles'].gt(data['vehicle_count']),
        'External score outside 0-100': (
            data['risk_score_external'].notna() & ~data['risk_score_external'].between(0, 100)),
        'Unexpected deductible': (
            data['deductible'].notna() & ~data['deductible'].isin([250, 500, 1000, 2500, 5000])),
    })
pd.DataFrame({name: flags.sum() for name, flags in numeric_issues.items()})


In [0]:
numeric_review = numeric_issues['Train'].any(axis=1) | train['vehicle_count'].eq(0)
train.loc[
    numeric_review,
    ['policy_id', 'vehicle_count', 'num_heavy_vehicles', 'driver_count', 'annual_premium']]

Three lines have more heavy vehicles than total vehicles. Preserve the original affected rows and mark both counts unknown on those three lines.

Five annual premiums are negative, so treat these amounts as unknown. Scoring has no flagged underwriting inputs. The resulting missing values will be imputed at the end.

In [0]:
numeric_review_rows = train.loc[numeric_review].copy()
fleet_conflict = numeric_issues['Train']['Heavy vehicles exceed fleet']
train.loc[fleet_conflict, ['vehicle_count', 'num_heavy_vehicles']] = np.nan
train.loc[train['annual_premium'].lt(0), 'annual_premium'] = np.nan

## Policy dates and snapshot timing

Parse the dates and distinguish originally missing values from nonempty strings that cannot be represented by the conversion.

In [0]:
date_columns = ['snapshot_date', 'policy_effective_date',
                'policy_expiration_date', 'first_claim_reported_date']
date_rows = []
for name, data in [('Train', train), ('Score', score)]:
    for col in date_columns:
        parsed = pd.to_datetime(data[col], format='%Y-%m-%d', errors='coerce')
        failed = data[col].notna() & parsed.isna()
        date_rows.append([name, col, data[col].isna().sum(), failed.sum(),
                          parsed.isna().sum(), data.loc[failed, col].value_counts().to_dict()])
        data[col] = parsed

pd.DataFrame(date_rows, columns=[
    'File', 'Column', 'Originally missing', 'Nonempty values converted to NaT',
    'Missing after conversion', 'Unconverted values'
])

The saved conversion output has 300 failures for the nonempty scoring value `9999-01-01`. Originally empty dates are counted separately.

The dictionary describes `first_claim_reported_date` as information available after claims occur. The field is excluded from inception predictors, and all scoring rows are retained.

In [0]:
date_observations = {}
for name, data in [('Train', train), ('Score', score)]:
    data['snapshot_year'] = data['snapshot_date'].dt.year
    data['policy_duration_days'] = (data['policy_expiration_date'] - data['policy_effective_date']).dt.days
    data['snapshot_delay_days'] = (data['snapshot_date'] - data['policy_effective_date']).dt.days
    date_observations[name] = pd.Series({
        'Expiration not after inception': data['policy_duration_days'].le(0).sum(),
        'Capture before inception': data['snapshot_delay_days'].lt(0).sum(),
        'Capture after expiration': data['snapshot_date'].gt(data['policy_expiration_date']).sum(),
        'First report before inception (review)': data['first_claim_reported_date'].lt(data['policy_effective_date']).sum(),
    })
pd.DataFrame(date_observations)

The ten captures after expiration occur on the same ten records whose expiration is before inception; we exclude those records for the impossible coverage interval.

In [0]:
invalid_policy_rows = train.loc[train['policy_duration_days'].le(0)].copy()
invalid_policy_rows[
    ['policy_id', 'policy_effective_date', 'policy_expiration_date', 'policy_duration_days']]

In [0]:
train = train.loc[train['policy_duration_days'].gt(0)].reset_index(drop=True)
score = score.reset_index(drop=True)
print(f'Valid-period training lines: {len(train):,}; scoring lines retained: {len(score):,}')

The ten training rows with -30-day intervals have expiration before inception. Keep them in invalid_policy_rows for investigation and exclude them from the modeling population. The remaining periods are all 365 days. Training now has 14,990 lines; scoring retains all 3,000 lines.

In [0]:
timing_summary = pd.DataFrame({
    name: data['snapshot_delay_days'].agg(['min', 'median', 'max'])
    for name, data in [('Train', train), ('Score', score)]
})
timing_summary

Snapshots occur 0-179 days after inception, with medians of 90 days in training and 89 in scoring. For the selected inception-time use case, retained predictors must represent underwriting information available at inception. Current-period claim fields describe later events and cannot be used to predict whether those events occur.

In [0]:
fig, ax = plt.subplots(figsize=(9, 4), dpi=200)
for name, data in [('Train', train), ('Score', score)]:
    counts = data['snapshot_year'].value_counts().sort_index()
    bars = ax.bar(counts.index, counts, label=name)
    ax.bar_label(bars, labels=[f'{n:,}' for n in counts], padding=4)

ax.set(title='Training and scoring snapshots',
       xlabel='Snapshot year', ylabel='Policy lines', ylim=(0, 5700))
ax.set_xticks(range(2020, 2025))
ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.2)

plt.tight_layout()
plt.show()
plt.close()

The snapshot plot shows when the records were captured: historical snapshots are in 2020-2022 and scoring snapshots are in 2023-2024. The actual inception and expiration dates remain in metadata for the preparation split.

## Numeric distributions and large values

Summarize the 14 usable numeric predictors after resolving the clearly conflicting inputs. The plots use observed values before imputation. Histograms use common bins and percentages within each file; missing values are omitted only from the relevant plot. Mileage is thousands of fleet miles, coverage limits are thousands of USD, and other monetary fields are USD.

In [0]:
numeric_summary = pd.concat({
    name: data[numeric_features].describe(percentiles=[0.5, 0.95]).T
    for name, data in [('Train', train), ('Score', score)]
}, names=['File', 'Input'])
numeric_summary.round(2)

In [0]:
panels = [('vehicle_count', 'Fleet size', 'Vehicles'),
          ('driver_avg_age', 'Average driver age', 'Years'),
          ('prior_year_mileage_000', 'Annual fleet mileage', 'Thousand miles'),
          ('risk_score_external', 'External risk assessment', 'Score: 0-100')]

fig, axes = plt.subplots(2, 2, figsize=(12, 8), dpi=200)
for ax, (col, title, unit) in zip(axes.flat, panels):
    combined = pd.concat([train[col], score[col]]).dropna()
    bins = np.arange(0.5, combined.max() + 1.5) if col == 'vehicle_count' else np.histogram_bin_edges(combined, bins=18)
    
    for name, data in [('Train', train), ('Score', score)]:
        values = data[col].dropna()
        ax.hist(values, bins=bins, weights=np.full(len(values), 100 / len(values)),
                histtype='step', linewidth=2, label=name)

    ax.set(title=title, xlabel=unit, ylabel='Observed policy values (%)')

    if col == 'vehicle_count':
        ax.set_xticks([1, 5, 10, 15, 20])
    
    ax.legend(frameon=False, fontsize=9)
    ax.grid(axis='y', alpha=0.2)

fig.suptitle('Fleet sizes; driver\'s age, fleet mileage', fontsize=15)
plt.tight_layout()
plt.show()
plt.close()

Fleet sizes center near eight vehicles in both files. Scoring drivers are younger on average: about 39 versus 42 years. Average scoring mileage is higher, about 231,500 versus 204,000 fleet miles. The external-score distribution is similar.

The next plots make the upper tails visible. Each box contains the central 50% of observed values, its line is the median, and dots beyond the 1.5-IQR whiskers are statistical outliers. Positive prior losses are shown to make their spread visible.

In [0]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), dpi=200)
for ax, col, title, unit in zip(axes,
        ['prior_year_mileage_000', 'prior_loss_amount', 'annual_premium'],
        ['Annual fleet mileage', 'Prior loss when positive', 'Annual premium'],
        ['Thousand miles', 'USD', 'USD']):
    samples = [data.loc[data[col].gt(0), col] for data in [train, score]]
    ax.boxplot(samples, flierprops={'markersize': 2, 'alpha': 0.3})
    ax.set_xticks([1, 2], ['Train', 'Score'])
    ax.set(title=title, ylabel=unit)
    ax.grid(axis='y', alpha=0.2)

fig.suptitle('Mileage, prior loss, and premium distributions', fontsize=15)
plt.tight_layout()
plt.show()
plt.close()

In [0]:
train.nlargest(5, 'prior_year_mileage_000')[
    ['policy_id', 'coverage_type', 'business_type', 'vehicle_count', 'prior_year_mileage_000']]

In [0]:
train.nlargest(5, 'prior_loss_amount')[
    ['policy_id', 'coverage_type', 'business_type', 'vehicle_count', 'prior_loss_amount']]

The largest stated mileage is 1,639,100 fleet miles on a 16-vehicle line, about 102,444 miles per vehicle. Prior loss reaches roughly USD 397,270. Premiums reach exactly USD 40,000 in both files.

## Claim occurrence and losses by coverage

Define `had_claim` from the historical claim count: 1 means at least one claim during the policy period, and 0 means none. `had_claim` will be the target y. `claim_count` and `total_loss_amount` describe outcomes; including them in X would reveal the event the model is meant to predict.

In [0]:
if train['claim_count'].isna().any() or train['claim_count'].lt(0).any():
    raise ValueError('Resolve missing or negative claim counts before defining the target.')

train['had_claim'] = train['claim_count'].gt(0).astype(int)
outcome_consistency = pd.Series({
    'Positive loss with zero claims': (train['claim_count'].eq(0) & train['total_loss_amount'].gt(0)).sum(),
    'Zero loss with positive claims': (train['claim_count'].gt(0) & train['total_loss_amount'].eq(0)).sum(),
})
outcome_consistency

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=200)
counts = train.had_claim.value_counts().reindex([0, 1])
bars = axes[0].bar(['No claim', 'At least one claim'], counts)
axes[0].bar_label(bars, labels=[f'{n:,}\n({n / len(train):.1%})' for n in counts], padding=4)
axes[0].set(title='Amount of claims',
            ylabel='Policy lines', ylim=(0, 15500))

positive_counts = train.loc[train['had_claim'].eq(1), 'claim_count'].value_counts().reindex(range(1, 8), fill_value=0)
bars = axes[1].bar(positive_counts.index, positive_counts)
axes[1].bar_label(bars, labels=[f'{n:,}' for n in positive_counts], padding=4)
axes[1].set(title='Number of claims',
            xlabel='Claims during the policy period', ylabel='Claim-positive lines', ylim=(0, 1850))
axes[1].set_xticks(range(1, 8))

for ax in axes:
    ax.grid(axis='y', alpha=0.2)

plt.tight_layout()
plt.show()
plt.close()

There are 1,853 claim-positive lines among 14,990 retained lines, a rate of 12.36%. The left plot uses all retained lines. The right plot uses only the 1,853 claim-positive lines and shows their claim counts; 1,616 had exactly one claim. The class imbalance will matter when evaluating a future model.

In [0]:
coverage_summary = train.groupby('coverage_type')['had_claim'] \
    .agg(Policies='size', Claim_positive='sum', Claim_rate='mean') \
    .reindex(['APD', 'AL'])

coverage_summary['Mean_loss_if_claim'] = train.loc[train.had_claim.eq(1)] \
    .groupby('coverage_type')['total_loss_amount'] \
    .mean()

coverage_summary.round(4)

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=200)
rates = 100 * coverage_summary['Claim_rate']
bars = axes[0].bar(rates.index, rates)
axes[0].bar_label(bars, fmt='%.1f%%', padding=4)
axes[0].set(title='Rate of claims within their coverage',
            ylabel='Lines with a claim within coverage (%)', ylim=(0, 20))

losses = coverage_summary['Mean_loss_if_claim']
bars = axes[1].bar(losses.index, losses)
axes[1].bar_label(bars, labels=[f'${value:,.0f}' for value in losses], padding=4)
axes[1].set(title='Mean losses by coverage',
            ylabel='Mean total loss on claim-positive lines (USD)', ylim=(0, 250000))

for ax in axes:
    ax.grid(axis='y', alpha=0.2)

plt.tight_layout()
plt.show()
plt.close()

In [0]:
print('Loss exceeds listed limit in USD:',
      train['total_loss_amount'].gt(1000 * train['coverage_limit_000']).sum())

In [0]:
train.nlargest(5, 'total_loss_amount')[
    ['policy_id', 'coverage_type', 'claim_count', 'total_loss_amount', 'coverage_limit_000']]

The maximum total loss is approximately USD 5.87 million on an AL line; 85 lines exceed their listed limit after converting the limit to USD.

The following comparison checks current-period claim information against `had_claim`.

In [0]:
current_claim_signals = pd.DataFrame({
    'Positive current paid amount': train['claim_paid_amount_current_period'].gt(0),
    'First claim date present': train['first_claim_reported_date'].notna(),
    'Status other than No Claim': train['claim_status_current_period'].ne('No Claim'),
})
current_claim_signals.eq(train['had_claim'].astype(bool), axis=0) \
    .mean() \
    .mul(100)

These three signals match `had_claim` on every retained historical line. They describe claims already observed.

## Claim rates by exposure and prior history

Compare claim occurrence across a few interpretable predictor groups. Separate APD and AL so their different baseline rates do not obscure other patterns. Each bar is the percentage of policy lines with a claim within that predictor group and coverage.

In [0]:
fleet_group = pd.cut(
    train.vehicle_count,
    [0, 5, 10, np.inf],
    labels=['1-5 vehicles', '6-10 vehicles', '11+ vehicles'])
fleet_rates = train.groupby([fleet_group, 'coverage_type'], observed=True)['had_claim'] \
    .mean() \
    .unstack()

fig, ax = plt.subplots(figsize=(9, 5), dpi=200)
fleet_rates.reindex(columns=['APD', 'AL']).mul(100).plot.bar(ax=ax, rot=0)

for bars in ax.containers:
    ax.bar_label(bars, fmt='%.1f%%', padding=4)

ax.set(title='At least 1 claim depending on fleet size',
       xlabel='Fleet size', ylabel='Policy lines with a claim (%)', ylim=(0, 32))
ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.2)

plt.tight_layout()
plt.show()
plt.close()

Observed claim rates rise with fleet size within both coverages.

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, coverage, column in zip(axes, ['APD', 'AL'], ['prior_apd_claim_count', 'prior_al_claim_count']):
    data = train.loc[train['coverage_type'].eq(coverage)]
    groups = data.groupby(data[column].clip(upper=3))['had_claim'].agg(['size', 'mean'])
    labels = [f'{"3+" if level == 3 else level}\n(n={n:,})' for level, n in groups['size'].items()]
    bars = ax.bar(labels, 100 * groups['mean'])
    ax.bar_label(bars, fmt='%.1f%%', padding=4)
    ax.set(title=f'{coverage}: prior claims and claim rate',
           xlabel=f'Prior-period {coverage} claims', ylabel='Policy lines with a claim (%)', ylim=(0, 36))
    ax.grid(axis='y', alpha=0.2)

plt.tight_layout()
plt.show()
plt.close()

More prior claims in the same coverage are associated with higher observed claim rates. The n values show the number of policy lines in each group. The AL group with three or more prior claims has only 35 lines, so its observed percentage is less stable.

In [0]:
score_group = pd.cut(
    train.risk_score_external,
    [0, 20, 40, 60, 80, 100],
    include_lowest=True,
    labels=['0-20', '20-40', '40-60', '60-80', '80-100'])
external_rates = train.groupby([score_group, 'coverage_type'], observed=True)['had_claim'] \
    .mean() \
    .unstack()

fig, ax = plt.subplots(figsize=(10, 5), dpi=200)
ax = external_rates.reindex(columns=['APD', 'AL']).mul(100).plot.bar(ax=ax, rot=0)

for bars in ax.containers:
    ax.bar_label(bars, fmt='%.1f%%', padding=4)

ax.set(title='External scores vs. claim rates',
       xlabel='External risk score: 0-100',
       ylabel='Policy lines with a claim (%)', ylim=(0, 27))

ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.show()
plt.close()

Higher external scores are associated with higher observed claim rates within coverage.

## Prepare predictors, target, and missing values

Keep 4 categorical and 14 numeric underwriting inputs in X, and `had_claim` in y.

The numeric columns `claim_count` and `total_loss_amount` are outcomes. Current-period claim/payment fields are unavailable at inception, `has_safety_program` is absent from scoring, and capture-derived columns are for inspection. Keep IDs and policy dates separately in metadata.

In [0]:
categorical_features = ['coverage_type', 'business_type', 'state', 'payment_frequency']
feature_columns = categorical_features + numeric_features
X = train[feature_columns].copy()
y = train['had_claim'].copy()
X_score = score[feature_columns].copy()

metadata_columns = ['policy_id', 'insured_id', 'policy_effective_date',
                    'policy_expiration_date', 'snapshot_date']
train_metadata = train[metadata_columns].copy()
score_metadata = score[metadata_columns].copy()

In [0]:
print('Training inputs:', X.shape, '| target:', y.shape)
print('Scoring inputs:', X_score.shape)
pd.DataFrame({
    'Train missing': X.isna().sum(),
    'Score missing': X_score.isna().sum()
}).loc[lambda d: d.gt(0).any(axis=1)]

## Fit preprocessing on an earlier training population

Missing-value handling happens after selecting predictors and separating training from validation. Use 2022 policy inceptions as a chronological holdout. For this preparation split, earlier training policies must end by 1 January 2022 so their full coverage periods precede validation. Policies spanning that cutoff are set aside from this split, not deleted from the retained 14,990 lines.

In [0]:
validation_start = pd.Timestamp('2022-01-01')
training_mask = train['policy_expiration_date'].le(validation_start)
validation_mask = train['policy_effective_date'].ge(validation_start)

X_train = X.loc[training_mask].copy()
y_train = y.loc[training_mask].copy()
X_valid = X.loc[validation_mask].copy()
y_valid = y.loc[validation_mask].copy()

print('Training partition:', len(X_train))
print('Validation partition:', len(X_valid))
print('Spanning the cutoff:', (~(training_mask | validation_mask)).sum())

## Fill unknown values and encode categories

Use training-partition medians for missing numeric predictors. This is a simple baseline for the skewed monetary and mileage fields and avoids treating unknown values as zero. Fill missing categorical values with Missing so the encoder keeps their absence distinct from known categories.

Fit the encoder on the same training partition, then reuse it for all other rows.

In [0]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [0]:
categorical_preprocessing = Pipeline([
    ('fill_missing', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('numeric', SimpleImputer(strategy='median'), numeric_features),
    ('categorical', categorical_preprocessing, categorical_features)
]).set_output(transform='pandas')

In [0]:
preprocessor.fit(X_train)
X_prepared = preprocessor.transform(X)
X_score_prepared = preprocessor.transform(X_score)

X_train_ready = X_prepared.loc[training_mask].copy()
X_valid_ready = X_prepared.loc[validation_mask].copy()

In [0]:
prepared_tables = {
    'Train': X_train_ready,
    'Validation': X_valid_ready,
    'All retained history': X_prepared,
    'Score': X_score_prepared
}

pd.DataFrame({
    name: {
        'Rows': len(data),
        'Numeric columns': data.shape[1],
        'Missing values': data.isna().sum().sum()
    }
    for name, data in prepared_tables.items()
}).T

In [0]:
X_train_ready.head()

## Old code

In [0]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [0]:
# ---- preprocessing ----

# fill numeric missing with 0 -- good enough for now
df2[df2.select_dtypes("number").columns] = df2.select_dtypes("number").fillna(0)
df3[df3.select_dtypes("number").columns] = df3.select_dtypes("number").fillna(0)

# encode categoricals for train
df2["coverage_type"] = df2["coverage_type"].astype("category").cat.codes
df2["business_type"]  = df2["business_type"].astype("category").cat.codes
df2["state"]          = df2["state"].astype("category").cat.codes

# same for score
df3["coverage_type"] = df3["coverage_type"].astype("category").cat.codes
df3["business_type"]  = df3["business_type"].astype("category").cat.codes
df3["state"]          = df3["state"].astype("category").cat.codes

In [0]:
# ---- target ----
# using binary had-a-claim flag as target
df2["target"] = (df2["claim_count"] > 0).astype(int)
print("positive rate:", round(df2["target"].mean(), 3))

In [0]:
# ---- features ----
feat_cols = [
    "coverage_type",
    "vehicle_count", "vehicle_avg_age", "driver_count", "driver_avg_age",
    "years_in_business", "prior_year_mileage_000",
    "business_type", "state",
    "prior_apd_claim_count", "prior_al_claim_count", "prior_loss_amount",
    "deductible", "coverage_limit_000", "annual_premium",
    "risk_score_external", "num_heavy_vehicles", "late_payment_count",
    "claim_paid_amount_current_period",
    "days_to_first_claim_report",
]

X = df2[feat_cols]
y = df2["target"]

In [0]:
# ---- fit model ----

# random split -- not time-based
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

tmp = LogisticRegression(max_iter=1000)
tmp.fit(X_train, y_train)

In [0]:
# ---- evaluate ----
x1 = tmp.predict(X_train)
print("accuracy:", accuracy_score(y_train, x1))
# results look solid

In [0]:
# ---- score new policies ----
X2 = df3[feat_cols]
preds = tmp.predict(X2)

df3["risk_score"] = preds
df3[["policy_id", "risk_score"]].to_csv("predictions.csv")
print("done -- predictions.csv written")